# LoRA From Scratch — Low-Rank Adaptation of Large Language Models

## What You'll Build
- ✅ Understand the **math** behind low-rank decomposition
- ✅ Implement `LoRALayer` from scratch (no PEFT library)
- ✅ Hook LoRA into **attention Q/K/V/O projections**
- ✅ Fine-tune a small GPT-2 style transformer on a custom task
- ✅ Compare trainable parameters: Full fine-tune vs LoRA
- ✅ Merge LoRA weights back into the base model
- ✅ Visualize rank decomposition and loss curves

---
## Background: Why LoRA?

Fine-tuning large models is expensive. GPT-3 has 175B parameters — updating all of them requires ~700GB of optimizer states.

**Key Insight (Hu et al., 2021):** Pre-trained weight matrices have low *intrinsic rank*. The updates during fine-tuning also lie in a low-rank subspace.

Instead of learning $\Delta W \in \mathbb{R}^{d \times k}$ (full rank), we learn:
$$\Delta W = BA$$
where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, and $r \ll \min(d, k)$.

**Parameter savings:** $(d \times k)$ → $(d \times r + r \times k) = r(d+k)$

For $d=k=4096$, $r=8$: $16M → 65K$ — a **246× reduction!**

---
## Section 1: Setup & Imports

In [ ]:
# Install dependencies
!pip install torch transformers datasets matplotlib numpy tqdm --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from copy import deepcopy
from tqdm import tqdm
import math
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

---
## Section 2: Understanding Low-Rank Decomposition

Before writing LoRA, let's understand *why* rank matters.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1 — Visualizing Low-Rank Approximation via SVD
# ─────────────────────────────────────────────────────────────────────────────

def low_rank_approx(W, rank):
    """Approximate matrix W with rank-r SVD decomposition."""
    U, S, Vt = torch.linalg.svd(W, full_matrices=False)
    # Keep only top-r singular values
    U_r = U[:, :rank]
    S_r = torch.diag(S[:rank])
    Vt_r = Vt[:rank, :]
    W_approx = U_r @ S_r @ Vt_r
    error = torch.norm(W - W_approx).item() / torch.norm(W).item()
    return W_approx, error

# Create a synthetic weight matrix (simulating a real weight update)
d, k = 128, 128
# Simulate a 'real' low-rank update: only rank-4 signal + noise
true_rank = 4
B_true = torch.randn(d, true_rank) * 0.1
A_true = torch.randn(true_rank, k) * 0.1
W_signal = B_true @ A_true          # true low-rank component
W_noise  = torch.randn(d, k) * 0.001  # tiny noise
W = W_signal + W_noise

# Compute SVD spectrum
_, S, _ = torch.linalg.svd(W)

# Approximate at different ranks
ranks = [1, 2, 4, 8, 16, 32, 64]
errors = [low_rank_approx(W, r)[1] for r in ranks]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Low-Rank Decomposition Intuition', fontsize=14, fontweight='bold')

# Singular value spectrum
axes[0].semilogy(S[:32].numpy(), 'b.-', markersize=8)
axes[0].axvline(x=true_rank-1, color='r', linestyle='--', label=f'True rank={true_rank}')
axes[0].set_xlabel('Singular Value Index')
axes[0].set_ylabel('Singular Value (log scale)')
axes[0].set_title('Singular Value Spectrum\n(energy drops sharply after rank 4)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reconstruction error vs rank
axes[1].plot(ranks, errors, 'ro-', markersize=8, linewidth=2)
axes[1].axvline(x=true_rank, color='b', linestyle='--', label=f'True rank={true_rank}')
axes[1].set_xlabel('Approximation Rank r')
axes[1].set_ylabel('Relative Frobenius Error')
axes[1].set_title('Reconstruction Error vs Rank')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Parameter count comparison
d_model, k_model = 768, 768  # GPT-2 small dimensions
full_params = d_model * k_model
lora_params = {r: r * (d_model + k_model) for r in [1, 2, 4, 8, 16, 32, 64]}
colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(lora_params)))
bars = axes[2].bar([str(r) for r in lora_params.keys()], 
                   [v/full_params*100 for v in lora_params.values()],
                   color=colors, edgecolor='black', linewidth=0.5)
axes[2].axhline(y=100, color='b', linestyle='--', linewidth=2, label=f'Full ({full_params:,} params)')
axes[2].set_xlabel('LoRA Rank r')
axes[2].set_ylabel('% of Full Parameters')
axes[2].set_title(f'LoRA Param Savings\n(d=k={d_model}, base={full_params:,} params)')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')
for bar, (r, v) in zip(bars, lora_params.items()):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v/full_params*100:.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('lora_intuition.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Parameter Comparison (d=k=768, GPT-2 small sized):")
print(f"{'Rank':<8} {'LoRA Params':<15} {'Full Params':<15} {'Savings':<10}")
print("-" * 50)
for r, v in lora_params.items():
    print(f"r={r:<6} {v:<15,} {full_params:<15,} {full_params/v:.0f}x")

---
## Section 3: LoRA Layer — From Scratch

### The Core Formula
For a pre-trained weight $W_0 \in \mathbb{R}^{d \times k}$, the forward pass becomes:

$$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} B A x$$

- $W_0$: **frozen** pre-trained weights
- $A \in \mathbb{R}^{r \times k}$: initialized with **Kaiming uniform** (random)
- $B \in \mathbb{R}^{d \times r}$: initialized with **zeros** (so $\Delta W = 0$ at init)
- $\alpha$: scaling hyperparameter (usually $\alpha = r$, so scale = 1)
- $r$: rank (hyperparameter, typically 4–64)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.1 — LoRALayer: The Core Building Block
# ─────────────────────────────────────────────────────────────────────────────

class LoRALayer(nn.Module):
    """
    LoRA low-rank adapter that wraps an existing nn.Linear layer.
    
    The forward pass computes:
        output = W0 @ x + scale * (B @ A) @ x
    
    where W0 is frozen and only A, B are trained.
    """
    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 8,
        alpha: float = 16.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.rank  = rank
        self.alpha = alpha
        self.scale = alpha / rank  # Scaling factor
        
        # ── Low-rank matrices ──────────────────────────────────────────────
        # A: (rank × in_features) — projects input DOWN to rank dimensions
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        # B: (out_features × rank) — projects back UP to output dimensions  
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        #
        # Why A random, B=zeros?
        # At init: ΔW = B @ A = 0 @ A = 0  →  no change to pretrained behavior!
        # Training starts from the exact pre-trained checkpoint.
        
        # Optional dropout on the low-rank path
        self.lora_dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        
        # Initialize A with Kaiming uniform (same as nn.Linear default)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        # B stays zero (already done above)
        
        print(f"  LoRALayer: ({out_features}×{in_features}) | "
              f"rank={rank} | params={rank*(in_features+out_features):,} | "
              f"scale={self.scale:.3f}")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., in_features)
        returns: (..., out_features)
        """
        # Low-rank path: x → A → (dropout) → B → scale
        # Shape trace: (..., in) @ (in, rank) = (..., rank)
        #              (..., rank) @ (rank, out) = (..., out)
        lora_out = self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T
        return self.scale * lora_out
    
    def get_delta_weight(self) -> torch.Tensor:
        """Return the full ΔW = scale * B @ A matrix (for merging/inspection)."""
        return self.scale * (self.lora_B @ self.lora_A)
    
    def extra_repr(self) -> str:
        return (f'in={self.in_features}, out={self.out_features}, '
                f'rank={self.rank}, alpha={self.alpha}, scale={self.scale:.3f}')


print("LoRALayer definition complete. Testing...")
print()

# Quick sanity check
lora = LoRALayer(in_features=64, out_features=64, rank=4, alpha=8.0)
x_test = torch.randn(2, 10, 64)  # (batch, seq, features)
out = lora(x_test)
print(f"\nInput shape:  {x_test.shape}")
print(f"Output shape: {out.shape}")
print(f"ΔW norm at init: {lora.get_delta_weight().norm():.6f}  ← should be ~0!")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.2 — LoRALinear: Drop-in Replacement for nn.Linear
# ─────────────────────────────────────────────────────────────────────────────

class LoRALinear(nn.Module):
    """
    A drop-in replacement for nn.Linear that adds a LoRA adapter.
    
    forward(x) = linear(x) + lora(x)
              = W0 @ x + b  +  scale * B @ A @ x
    
    The original weight W0 is FROZEN; only lora_A and lora_B train.
    """
    def __init__(
        self,
        linear: nn.Linear,      # The existing pretrained layer to wrap
        rank: int = 8,
        alpha: float = 16.0,
        dropout: float = 0.0,
        enabled: bool = True,   # Can toggle LoRA on/off
    ):
        super().__init__()
        self.linear  = linear
        self.enabled = enabled
        
        # Freeze the original weights
        for param in self.linear.parameters():
            param.requires_grad = False
        
        # Attach LoRA adapter
        if enabled:
            self.lora = LoRALayer(
                in_features  = linear.in_features,
                out_features = linear.out_features,
                rank=rank, alpha=alpha, dropout=dropout
            )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.linear(x)
        if self.enabled:
            base_out = base_out + self.lora(x)
        return base_out
    
    def merge_weights(self) -> nn.Linear:
        """
        Merge LoRA weights into the base linear layer.
        After merging: W_merged = W0 + scale * B @ A
        This allows zero-overhead inference (no extra computation).
        """
        if not self.enabled:
            return self.linear
        
        merged = deepcopy(self.linear)
        delta_w = self.lora.get_delta_weight()  # (out, in)
        merged.weight.data = merged.weight.data + delta_w
        merged.weight.requires_grad = True  # Unfreeze for future use
        return merged
    
    def lora_parameters(self):
        """Return only LoRA parameters (for selective optimization)."""
        if self.enabled:
            return list(self.lora.parameters())
        return []
    
    def count_parameters(self):
        base = sum(p.numel() for p in self.linear.parameters())
        lora = sum(p.numel() for p in self.lora.parameters()) if self.enabled else 0
        return {'base': base, 'lora': lora, 'total': base + lora}


# Test LoRALinear
print("\n" + "="*60)
print("Testing LoRALinear wrapper")
print("="*60)

base_linear = nn.Linear(64, 64)
lora_linear = LoRALinear(base_linear, rank=4, alpha=8.0)

x = torch.randn(2, 10, 64)
out_base  = base_linear(x)
out_lora  = lora_linear(x)

print(f"\nBase output norm:  {out_base.norm():.4f}")
print(f"LoRA output norm:  {out_lora.norm():.4f}")
print(f"Difference at init: {(out_base - out_lora).norm():.6f}  ← should be ~0!")

params = lora_linear.count_parameters()
print(f"\nParameter counts:")
print(f"  Base (frozen): {params['base']:,}")
print(f"  LoRA (trainable): {params['lora']:,}")
print(f"  Savings: {params['base']/params['lora']:.1f}x fewer trainable params")

---
## Section 4: Building a Transformer with LoRA in Attention

Now we'll build a GPT-2 style transformer and inject LoRA into the **Q, K, V, O** projections of multi-head attention — the most impactful place for LoRA.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.1 — Multi-Head Attention with LoRA hooks
# ─────────────────────────────────────────────────────────────────────────────

class MultiHeadAttentionWithLoRA(nn.Module):
    """
    Multi-Head Self-Attention with LoRA adapters on Q, K, V, O projections.
    
    The LoRA modifications:
        Q_out = (W_Q + ΔW_Q) x = W_Q x + scale * B_Q A_Q x
        K_out = (W_K + ΔW_K) x  (optional, often skipped for efficiency)
        V_out = (W_V + ΔW_V) x
        O_out = (W_O + ΔW_O) x
    """
    def __init__(
        self,
        d_model:   int,
        n_heads:   int,
        rank:      int   = 8,
        alpha:     float = 16.0,
        dropout:   float = 0.1,
        lora_targets: tuple = ('q', 'v'),  # Which projections to LoRA-ify
    ):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model   = d_model
        self.n_heads   = n_heads
        self.head_dim  = d_model // n_heads
        self.scale     = self.head_dim ** -0.5
        self.lora_targets = lora_targets
        
        # Create base projection layers
        W_q = nn.Linear(d_model, d_model, bias=False)
        W_k = nn.Linear(d_model, d_model, bias=False)
        W_v = nn.Linear(d_model, d_model, bias=False)
        W_o = nn.Linear(d_model, d_model, bias=False)
        
        # Wrap with LoRA where specified
        lora_kwargs = dict(rank=rank, alpha=alpha, dropout=dropout)
        
        self.W_q = LoRALinear(W_q, enabled='q' in lora_targets, **lora_kwargs)
        self.W_k = LoRALinear(W_k, enabled='k' in lora_targets, **lora_kwargs)
        self.W_v = LoRALinear(W_v, enabled='v' in lora_targets, **lora_kwargs)
        self.W_o = LoRALinear(W_o, enabled='o' in lora_targets, **lora_kwargs)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        
    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor = None,
    ) -> torch.Tensor:
        B, T, C = x.shape  # (batch, seq_len, d_model)
        
        # ── Project to Q, K, V ─────────────────────────────────────────────
        # Each shape: (B, T, d_model)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # ── Reshape for multi-head: (B, n_heads, T, head_dim) ──────────────
        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        # ── Scaled dot-product attention ────────────────────────────────────
        scores = (Q @ K.transpose(-2, -1)) * self.scale  # (B, H, T, T)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        
        # Weighted sum of values
        out = attn_weights @ V  # (B, H, T, head_dim)
        
        # ── Concatenate heads and project out ──────────────────────────────
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, d_model)
        out = self.resid_dropout(self.W_o(out))
        
        return out, attn_weights
    
    def lora_parameters(self):
        """Collect all LoRA parameters across Q, K, V, O."""
        params = []
        for proj in [self.W_q, self.W_k, self.W_v, self.W_o]:
            params.extend(proj.lora_parameters())
        return params


# Test it
print("\n" + "="*60)
print("Multi-Head Attention with LoRA on Q and V")
print("="*60)
mha = MultiHeadAttentionWithLoRA(d_model=64, n_heads=4, rank=4, alpha=8.0, lora_targets=('q','v'))
x = torch.randn(2, 10, 64)
out, attn = mha(x)
print(f"\nInput:   {x.shape}")
print(f"Output:  {out.shape}")
print(f"Attn:    {attn.shape}")
lora_params = mha.lora_parameters()
print(f"LoRA trainable params: {sum(p.numel() for p in lora_params):,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.2 — Full Transformer Block and GPT-style Model
# ─────────────────────────────────────────────────────────────────────────────

class FeedForward(nn.Module):
    """Standard transformer FFN: Linear → GELU → Linear."""
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-norm transformer block with LoRA in attention."""
    def __init__(self, d_model, n_heads, d_ff, rank=8, alpha=16.0,
                 dropout=0.1, lora_targets=('q','v')):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttentionWithLoRA(
            d_model, n_heads, rank=rank, alpha=alpha,
            dropout=dropout, lora_targets=lora_targets
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = FeedForward(d_model, d_ff, dropout)
    
    def forward(self, x, mask=None):
        # Pre-norm residual connections
        attn_out, attn_weights = self.attn(self.norm1(x), mask)
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x, attn_weights
    
    def lora_parameters(self):
        return self.attn.lora_parameters()


class MiniGPTWithLoRA(nn.Module):
    """
    A small GPT-style language model with LoRA adapters.
    
    Architecture:
        Token Embedding + Positional Embedding
        → N × TransformerBlock (with LoRA in attention)
        → LayerNorm
        → Linear head (to vocab)
    """
    def __init__(
        self,
        vocab_size: int,
        max_seq_len: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        rank: int = 8,
        alpha: float = 16.0,
        dropout: float = 0.1,
        lora_targets: tuple = ('q', 'v'),
    ):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        
        # Embeddings (frozen during LoRA fine-tuning)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model, n_heads, d_ff,
                rank=rank, alpha=alpha,
                dropout=dropout, lora_targets=lora_targets
            )
            for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Tie weights (standard GPT practice)
        self.head.weight = self.tok_emb.weight
        
        self._init_weights()
        print(f"\nModel created: {self.count_parameters()['total']:,} total params")
    
    def _init_weights(self):
        """GPT-2 style weight init."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
    
    def get_causal_mask(self, seq_len: int, device):
        """Upper-triangular mask for causal (autoregressive) attention."""
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        return mask.view(1, 1, seq_len, seq_len)  # (1, 1, T, T)
    
    def forward(
        self,
        input_ids: torch.Tensor,  # (B, T)
        targets: torch.Tensor = None,  # (B, T) for LM loss
    ):
        B, T = input_ids.shape
        assert T <= self.max_seq_len, f"Sequence too long: {T} > {self.max_seq_len}"
        
        # Build embeddings
        pos   = torch.arange(T, device=input_ids.device)
        x     = self.drop(self.tok_emb(input_ids) + self.pos_emb(pos))
        mask  = self.get_causal_mask(T, input_ids.device)
        
        # Pass through transformer blocks
        all_attn = []
        for block in self.blocks:
            x, attn = block(x, mask)
            all_attn.append(attn)
        
        x      = self.norm(x)
        logits = self.head(x)  # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            # Shift: predict next token
            loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
                targets[:, 1:].contiguous().view(-1),
                ignore_index=-100
            )
        
        return logits, loss, all_attn
    
    def freeze_base_model(self):
        """Freeze everything except LoRA parameters and head."""
        for name, param in self.named_parameters():
            param.requires_grad = False
        
        # Unfreeze only LoRA params
        for block in self.blocks:
            for param in block.lora_parameters():
                param.requires_grad = True
        
        # Optionally unfreeze final layer norm and head
        for param in self.norm.parameters():
            param.requires_grad = True
    
    def count_parameters(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        lora_only = sum(
            p.numel() for block in self.blocks
            for p in block.lora_parameters()
        )
        return {'total': total, 'trainable': trainable, 'lora': lora_only}
    
    def print_parameter_table(self):
        counts = self.count_parameters()
        print("\n" + "="*55)
        print(" Parameter Count Summary")
        print("="*55)
        print(f"  Total parameters:      {counts['total']:>12,}")
        print(f"  Trainable parameters:  {counts['trainable']:>12,}")
        print(f"  LoRA-only parameters:  {counts['lora']:>12,}")
        if counts['total'] > 0:
            pct = 100 * counts['trainable'] / counts['total']
            print(f"  Training ratio:        {pct:>11.2f}%")
        print("="*55)


# ─── Build a small model ────────────────────────────────────────────────────
CONFIG = dict(
    vocab_size  = 256,   # Character-level: 256 byte tokens
    max_seq_len = 128,
    d_model     = 128,
    n_heads     = 4,
    n_layers    = 4,
    d_ff        = 512,
    rank        = 8,
    alpha       = 16.0,
    dropout     = 0.1,
    lora_targets= ('q', 'v'),  # Standard choice: Q and V only
)

print("Building MiniGPT with LoRA...")
model = MiniGPTWithLoRA(**CONFIG).to(DEVICE)
model.print_parameter_table()

# Now freeze base and show trainable-only count
print("\nAfter freezing base model (LoRA fine-tuning mode):")
model.freeze_base_model()
model.print_parameter_table()

---
## Section 5: Injecting LoRA into an Existing Model

In practice, you load a pre-trained checkpoint and *inject* LoRA adapters post-hoc — without touching the model's source code.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.1 — LoRA Injection: Patching an existing model's Linear layers
# ─────────────────────────────────────────────────────────────────────────────

def inject_lora(
    model: nn.Module,
    target_modules: list,  # e.g. ['q_proj', 'v_proj', 'W_q', 'W_v']
    rank: int = 8,
    alpha: float = 16.0,
    dropout: float = 0.0,
    verbose: bool = True,
) -> nn.Module:
    """
    Recursively walks model and replaces matching nn.Linear layers
    with LoRALinear wrappers.
    
    This is how real LoRA libraries (PEFT, LoRAX) work internally.
    """
    replaced = 0
    
    def _inject(module, prefix=''):
        nonlocal replaced
        for name, child in list(module.named_children()):
            full_name = f"{prefix}.{name}" if prefix else name
            
            # Check if this module name matches our targets
            if any(target in full_name for target in target_modules):
                if isinstance(child, nn.Linear):
                    if verbose:
                        print(f"  Injecting LoRA into: {full_name} "
                              f"({child.in_features}→{child.out_features})")
                    # Replace with LoRALinear
                    new_layer = LoRALinear(child, rank=rank, alpha=alpha, dropout=dropout)
                    setattr(module, name, new_layer)
                    replaced += 1
                    continue
            
            # Recurse into children
            _inject(child, full_name)
    
    _inject(model)
    
    # Freeze everything, then unfreeze LoRA params
    for param in model.parameters():
        param.requires_grad = False
    
    lora_count = 0
    for module in model.modules():
        if isinstance(module, LoRALayer):
            for p in module.parameters():
                p.requires_grad = True
                lora_count += p.numel()
    
    if verbose:
        total = sum(p.numel() for p in model.parameters())
        print(f"\n✅ Injected LoRA into {replaced} layers")
        print(f"   Trainable: {lora_count:,} / {total:,} ({100*lora_count/total:.2f}%)")
    
    return model


# Demonstrate on a fresh model (simulating loading a pretrained checkpoint)
print("Simulating LoRA injection into a pretrained model...\n")

# Create a 'pretrained' model
class SimpleTransformer(nn.Module):
    """Minimal transformer to demonstrate injection."""
    def __init__(self, d=64, n_heads=4, vocab=100):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.q_proj = nn.Linear(d, d)   # ← target
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)   # ← target
        self.o_proj = nn.Linear(d, d)
        self.mlp1   = nn.Linear(d, d*4)
        self.mlp2   = nn.Linear(d*4, d)
        self.head   = nn.Linear(d, vocab)
    def forward(self, x): return self.head(self.emb(x))

pretrained = SimpleTransformer()
print(f"Before injection: {sum(p.numel() for p in pretrained.parameters() if p.requires_grad):,} trainable params")

# Inject LoRA only into q_proj and v_proj
pretrained = inject_lora(
    pretrained,
    target_modules=['q_proj', 'v_proj'],
    rank=4,
    alpha=8.0,
)

---
## Section 6: Fine-Tuning with LoRA

We'll fine-tune our MiniGPT on a character-level task: learning to generate text in a specific style.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.1 — Dataset: Character-level text
# ─────────────────────────────────────────────────────────────────────────────

# Simple synthetic task: learn repetition patterns & structure
TRAINING_TEXT = """
The transformer architecture revolutionized natural language processing.
Attention mechanisms allow models to focus on relevant parts of the input.
Self-attention computes relationships between all positions simultaneously.
Multi-head attention learns different types of relationships in parallel.
Feed-forward networks add non-linearity between attention layers.
Layer normalization stabilizes training in deep transformer networks.
Positional encodings give the model information about token positions.
The transformer uses residual connections to ease gradient flow.
Pre-training on large corpora gives models general language understanding.
Fine-tuning adapts pre-trained models to specific downstream tasks.
LoRA adds low-rank matrices to frozen pre-trained weights.
Only the low-rank matrices are trained during fine-tuning.
This dramatically reduces the number of trainable parameters.
LoRA achieves comparable performance to full fine-tuning.
The rank controls the expressivity of the low-rank adaptation.
Higher rank captures more complex adaptations but uses more parameters.
The alpha parameter scales the contribution of the LoRA update.
LoRA matrices are initialized so the update starts at zero.
After training, LoRA weights can be merged with the base model.
Merged models have identical inference cost to the original.
""" * 20  # Repeat to have enough training data


class CharDataset(torch.utils.data.Dataset):
    """Character-level dataset — tokenizes as raw bytes (0-255)."""
    def __init__(self, text: str, seq_len: int):
        self.seq_len = seq_len
        # Encode as bytes
        data = list(text.encode('utf-8'))
        self.data = torch.tensor(data, dtype=torch.long)
        print(f"Dataset: {len(self.data):,} tokens, {len(self)} sequences")
    
    def __len__(self):
        return max(0, len(self.data) - self.seq_len)
    
    def __getitem__(self, idx):
        chunk = self.data[idx : idx + self.seq_len + 1]
        x = chunk[:-1]  # input
        y = chunk[1:]   # target (next token)
        return x, y


SEQ_LEN = 64
BATCH_SIZE = 32

dataset = CharDataset(TRAINING_TEXT, seq_len=SEQ_LEN)
loader  = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.2 — Build models: Base (full fine-tune) vs LoRA
# ─────────────────────────────────────────────────────────────────────────────

def build_model(use_lora=True, rank=8, lora_targets=('q','v')):
    cfg = dict(
        vocab_size   = 256,
        max_seq_len  = SEQ_LEN,
        d_model      = 128,
        n_heads      = 4,
        n_layers     = 4,
        d_ff         = 512,
        rank         = rank,
        alpha        = rank * 2.0,
        dropout      = 0.1,
        lora_targets = lora_targets,
    )
    m = MiniGPTWithLoRA(**cfg).to(DEVICE)
    if use_lora:
        m.freeze_base_model()  # Freeze base; train only LoRA + norm
    return m


print("\n" + "="*60)
print(" Model A: Full Fine-Tuning (all params trainable)")
print("="*60)
model_full = build_model(use_lora=False)
model_full.print_parameter_table()

print("\n" + "="*60)
print(" Model B: LoRA Fine-Tuning (rank=8, Q+V only)")
print("="*60)
model_lora = build_model(use_lora=True, rank=8)
model_lora.print_parameter_table()

print("\n" + "="*60)
print(" Model C: LoRA Fine-Tuning (rank=4, Q+V only) — even lighter")
print("="*60)
model_lora_r4 = build_model(use_lora=True, rank=4)
model_lora_r4.print_parameter_table()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.3 — Training Loop
# ─────────────────────────────────────────────────────────────────────────────

def train(
    model,
    loader,
    n_epochs=10,
    lr=3e-4,
    label='',
    use_lora=True,
):
    # Only optimize trainable parameters
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs * len(loader)
    )
    
    model.train()
    history = []
    
    print(f"\nTraining {label}...")
    print(f"  Trainable params: {sum(p.numel() for p in trainable):,}")
    print(f"  Epochs: {n_epochs} | LR: {lr}")
    
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        n_batches  = 0
        
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            
            optimizer.zero_grad()
            _, loss, _ = model(x, targets=x)  # LM loss: predict next token
            loss.backward()
            
            # Gradient clipping (standard practice)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            
            optimizer.step()
            scheduler.step()
            
            epoch_loss += loss.item()
            n_batches  += 1
        
        avg_loss = epoch_loss / n_batches
        history.append(avg_loss)
        
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.4f} | "
                  f"PPL: {math.exp(avg_loss):.2f}")
    
    return history


N_EPOCHS = 15

# Train full fine-tune model
history_full = train(
    model_full, loader,
    n_epochs=N_EPOCHS, lr=3e-4,
    label='Full Fine-Tuning', use_lora=False
)

# Train LoRA model (rank=8)
history_lora = train(
    model_lora, loader,
    n_epochs=N_EPOCHS, lr=3e-4,
    label='LoRA (rank=8)', use_lora=True
)

# Train LoRA model (rank=4)
history_lora_r4 = train(
    model_lora_r4, loader,
    n_epochs=N_EPOCHS, lr=3e-4,
    label='LoRA (rank=4)', use_lora=True
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4 — Plot Training Curves
# ─────────────────────────────────────────────────────────────────────────────

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('LoRA vs Full Fine-Tuning', fontsize=14, fontweight='bold')

epochs = range(1, N_EPOCHS + 1)

# Loss curves
axes[0].plot(epochs, history_full,     'b-o', label='Full Fine-tune', linewidth=2, markersize=5)
axes[0].plot(epochs, history_lora,     'r-s', label='LoRA (rank=8)',  linewidth=2, markersize=5)
axes[0].plot(epochs, history_lora_r4,  'g-^', label='LoRA (rank=4)',  linewidth=2, markersize=5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Trainable parameter comparison
n_full  = count_trainable(model_full)
n_lora8 = count_trainable(model_lora)
n_lora4 = count_trainable(model_lora_r4)

labels = ['Full\nFine-tune', 'LoRA\n(rank=8)', 'LoRA\n(rank=4)']
counts = [n_full, n_lora8, n_lora4]
colors = ['steelblue', 'tomato', 'mediumseagreen']
bars = axes[1].bar(labels, counts, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
axes[1].set_ylabel('Trainable Parameters')
axes[1].set_title('Trainable Parameter Count')
axes[1].grid(True, alpha=0.3, axis='y')

for bar, count in zip(bars, counts):
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + max(counts)*0.01,
        f'{count:,}\n({100*count/n_full:.1f}%)',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

plt.tight_layout()
plt.savefig('lora_training_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Loss Comparison:")
print(f"  Full fine-tune: {history_full[-1]:.4f}")
print(f"  LoRA (rank=8):  {history_lora[-1]:.4f}")
print(f"  LoRA (rank=4):  {history_lora_r4[-1]:.4f}")

---
## Section 7: Merging LoRA Weights

After training, we can **merge** LoRA weights back into the base model:
$$W_{merged} = W_0 + \frac{\alpha}{r} BA$$

This gives **identical inference**, but with **zero extra computation** — no LoRA layers needed.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.1 — Merging and verifying correctness
# ─────────────────────────────────────────────────────────────────────────────

def merge_lora_weights(model: nn.Module, verbose: bool = True) -> nn.Module:
    """
    Walk the model and merge all LoRALinear layers back into plain nn.Linear.
    Returns a new model with merged weights and no LoRA overhead.
    """
    merged_model = deepcopy(model)
    n_merged = 0
    
    def _merge(module):
        nonlocal n_merged
        for name, child in list(module.named_children()):
            if isinstance(child, LoRALinear) and child.enabled:
                # Replace with merged linear
                merged_linear = child.merge_weights()
                setattr(module, name, merged_linear)
                n_merged += 1
            else:
                _merge(child)
    
    _merge(merged_model)
    
    if verbose:
        print(f"✅ Merged {n_merged} LoRA layers into base weights")
        n_params = sum(p.numel() for p in merged_model.parameters())
        n_train  = sum(p.numel() for p in merged_model.parameters() if p.requires_grad)
        print(f"   Total params:     {n_params:,}")
        print(f"   Trainable params: {n_train:,}")
    
    return merged_model


# Merge the LoRA model
print("Merging LoRA weights...")
model_merged = merge_lora_weights(model_lora)

# Verify: merged model should produce same output as LoRA model
model_lora.eval()
model_merged.eval()

with torch.no_grad():
    test_input = torch.randint(0, 256, (2, SEQ_LEN)).to(DEVICE)
    
    logits_lora,   _, _ = model_lora(test_input)
    logits_merged, _, _ = model_merged(test_input)
    
    max_diff = (logits_lora - logits_merged).abs().max().item()
    mean_diff = (logits_lora - logits_merged).abs().mean().item()

print(f"\nVerification:")
print(f"  Max logit difference:  {max_diff:.2e}  ← should be ~0 (numerical noise only)")
print(f"  Mean logit difference: {mean_diff:.2e}")
print(f"  Outputs match: {'✅ YES' if max_diff < 1e-4 else '❌ NO'}")

---
## Section 8: Visualizing What LoRA Learns

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.1 — Visualize LoRA weight matrices and singular value spectra
# ─────────────────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(20, 12))
fig.suptitle('LoRA Weight Analysis After Training', fontsize=14, fontweight='bold')

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.35)

row_labels = ['Layer 0', 'Layer 1', 'Layer 2', 'Layer 3']

# Collect LoRA matrices from all blocks
all_delta_W = []
all_spectra  = []

for i, block in enumerate(model_lora.blocks):
    # Get Q and V LoRA adapters
    W_q_lora = block.attn.W_q
    W_v_lora = block.attn.W_v
    
    for proj, proj_name in [(W_q_lora, 'Q'), (W_v_lora, 'V')]:
        if proj.enabled:
            dW = proj.lora.get_delta_weight().detach().cpu()
            all_delta_W.append((f'L{i}-{proj_name}', dW))
            # SVD of delta W
            _, S, _ = torch.linalg.svd(dW)
            all_spectra.append((f'L{i}-{proj_name}', S))

# Plot 1: ΔW heatmaps (first 4)
ax_titles = []
for idx in range(min(4, len(all_delta_W))):
    ax = fig.add_subplot(gs[0, idx])
    name, dW = all_delta_W[idx]
    im = ax.imshow(dW.numpy()[:32, :32], cmap='RdBu', aspect='auto',
                   vmin=-dW.abs().max(), vmax=dW.abs().max())
    ax.set_title(f'ΔW ({name})\n[first 32×32]', fontsize=9)
    ax.set_xlabel('in_features')
    ax.set_ylabel('out_features')
    plt.colorbar(im, ax=ax, fraction=0.046)

# Plot 2: Singular value spectra
ax_spec = fig.add_subplot(gs[1, :2])
colors_spec = plt.cm.tab10(np.linspace(0, 1, len(all_spectra)))
for (name, S), c in zip(all_spectra, colors_spec):
    rank = len([s for s in S if s > 1e-6])
    ax_spec.plot(S[:16].numpy(), '.-', color=c, label=f'{name} (eff.rank≈{rank})', linewidth=1.5)
ax_spec.set_xlabel('Singular Value Index')
ax_spec.set_ylabel('Singular Value')
ax_spec.set_title('Singular Value Spectrum of ΔW\n(lower = more compressed representation)')
ax_spec.legend(fontsize=8, ncol=2)
ax_spec.grid(True, alpha=0.3)

# Plot 3: A and B matrix norms per layer
ax_norm = fig.add_subplot(gs[1, 2:])
A_norms, B_norms, layer_names = [], [], []
for i, block in enumerate(model_lora.blocks):
    for proj, pname in [(block.attn.W_q, 'Q'), (block.attn.W_v, 'V')]:
        if proj.enabled:
            A_norms.append(proj.lora.lora_A.detach().norm().item())
            B_norms.append(proj.lora.lora_B.detach().norm().item())
            layer_names.append(f'L{i}-{pname}')

x_pos = np.arange(len(layer_names))
width = 0.35
ax_norm.bar(x_pos - width/2, A_norms, width, label='||A|| (down-proj)', color='steelblue', alpha=0.8)
ax_norm.bar(x_pos + width/2, B_norms, width, label='||B|| (up-proj)',   color='tomato',    alpha=0.8)
ax_norm.set_xticks(x_pos)
ax_norm.set_xticklabels(layer_names, rotation=45, ha='right', fontsize=9)
ax_norm.set_ylabel('Frobenius Norm')
ax_norm.set_title('LoRA Matrix Norms\n(B starts at 0, A starts random)')
ax_norm.legend()
ax_norm.grid(True, alpha=0.3, axis='y')

# Plot 4: ΔW magnitude vs layer (effective adaptation)
ax_mag = fig.add_subplot(gs[2, :2])
delta_norms = []
delta_labels = []
for name, dW in all_delta_W:
    delta_norms.append(dW.norm().item())
    delta_labels.append(name)

ax_mag.bar(delta_labels, delta_norms, color=plt.cm.viridis(np.linspace(0.2,0.8,len(delta_labels))),
           edgecolor='black', linewidth=0.5)
ax_mag.set_xlabel('Layer-Projection')
ax_mag.set_ylabel('||ΔW|| (Frobenius)')
ax_mag.set_title('Learned Update Magnitude per Layer\n(larger = more adaptation here)')
ax_mag.tick_params(axis='x', rotation=45)
ax_mag.grid(True, alpha=0.3, axis='y')

# Plot 5: Parameter efficiency visualization  
ax_pie = fig.add_subplot(gs[2, 2:])
total = sum(p.numel() for p in model_lora.parameters())
lora_p = sum(p.numel() for block in model_lora.blocks for p in block.lora_parameters())
frozen = total - lora_p

wedge_sizes = [frozen, lora_p]
wedge_labels = [f'Frozen\n({frozen:,}\nparams)', f'LoRA\n({lora_p:,}\nparams)']
wedge_colors = ['lightgrey', 'tomato']
wedge_explode = [0, 0.05]
ax_pie.pie(wedge_sizes, labels=wedge_labels, colors=wedge_colors,
           explode=wedge_explode, autopct='%1.1f%%', startangle=90,
           textprops={'fontsize': 10})
ax_pie.set_title(f'Parameter Distribution\n(rank=8, Q+V adaptation)')

plt.savefig('lora_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.2 — Text Generation: LoRA model vs Base
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def generate(model, prompt: str, max_new_tokens=100, temperature=0.8, top_k=50):
    """Autoregressive text generation with top-k sampling."""
    model.eval()
    
    # Encode prompt
    ids = list(prompt.encode('utf-8'))
    ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    
    for _ in range(max_new_tokens):
        # Crop to max_seq_len
        ids_crop = ids[:, -SEQ_LEN:]
        
        # Get logits
        logits, _, _ = model(ids_crop)
        logits = logits[:, -1, :] / temperature  # Last token logits
        
        # Top-k filtering
        if top_k > 0:
            topk_vals = torch.topk(logits, top_k)
            logits[logits < topk_vals.values[:, -1:]] = float('-inf')
        
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    
    # Decode
    generated = ids[0, len(prompt.encode('utf-8')):].tolist()
    try:
        return bytes(generated).decode('utf-8', errors='replace')
    except:
        return str(generated)


PROMPT = "The transformer architecture"

print("=" * 60)
print("Text Generation Samples")
print("=" * 60)

print(f"\nPrompt: '{PROMPT}'")
print()

for label, m in [
    ('Full Fine-tune', model_full),
    ('LoRA (rank=8)',  model_lora),
    ('Merged LoRA',   model_merged),
]:
    gen = generate(m, PROMPT, max_new_tokens=80, temperature=0.7)
    print(f"[{label}]")
    print(f"  {PROMPT}{gen}")
    print()

---
## Section 9: LoRA Ablation Study

Explore the effect of different ranks and which projections to adapt.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9.1 — Rank Ablation
# ─────────────────────────────────────────────────────────────────────────────

ABLATION_EPOCHS = 8  # Fewer epochs for speed
ranks_to_test = [1, 2, 4, 8, 16]

ablation_results = {}

for rank in ranks_to_test:
    print(f"\n--- Training LoRA rank={rank} ---")
    m = build_model(use_lora=True, rank=rank)
    hist = train(m, loader, n_epochs=ABLATION_EPOCHS, lr=3e-4, label=f'rank={rank}')
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    ablation_results[rank] = {
        'history': hist,
        'final_loss': hist[-1],
        'params': n_params,
    }

# ── Plot ablation results ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('LoRA Rank Ablation Study', fontsize=14, fontweight='bold')

colors = plt.cm.cool(np.linspace(0.1, 0.9, len(ranks_to_test)))

# Loss curves
for rank, color in zip(ranks_to_test, colors):
    hist = ablation_results[rank]['history']
    axes[0].plot(range(1, len(hist)+1), hist, '.-', color=color,
                label=f'r={rank}', linewidth=2, markersize=6)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss vs Epoch by Rank')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Final loss vs rank
ranks_arr = list(ranks_to_test)
final_losses = [ablation_results[r]['final_loss'] for r in ranks_arr]
axes[1].plot(ranks_arr, final_losses, 'o-', color='tomato', linewidth=2, markersize=10)
axes[1].set_xlabel('Rank r')
axes[1].set_ylabel('Final Loss')
axes[1].set_title('Final Loss vs Rank')
axes[1].grid(True, alpha=0.3)
for r, l in zip(ranks_arr, final_losses):
    axes[1].annotate(f'{l:.3f}', (r, l), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=9)

# Params vs rank (tradeoff curve)
params_arr = [ablation_results[r]['params'] for r in ranks_arr]
axes[2].scatter(params_arr, final_losses, c=colors, s=200, zorder=5, edgecolors='black')
axes[2].plot(params_arr, final_losses, '--', color='gray', alpha=0.5)
for r, (p, l) in zip(ranks_arr, zip(params_arr, final_losses)):
    axes[2].annotate(f'r={r}\n({p:,})', (p, l), textcoords='offset points',
                    xytext=(5, 5), fontsize=8)
axes[2].set_xlabel('Trainable Parameters')
axes[2].set_ylabel('Final Loss')
axes[2].set_title('Efficiency Frontier:\nLoss vs Parameter Count')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lora_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Rank Ablation Summary:")
print(f"{'Rank':<8} {'Params':<12} {'Final Loss':<12} {'vs r=1':<10}")
print("-" * 45)
base_loss = ablation_results[1]['final_loss']
for r in ranks_to_test:
    res = ablation_results[r]
    improvement = (base_loss - res['final_loss']) / base_loss * 100
    print(f"r={r:<6} {res['params']:<12,} {res['final_loss']:<12.4f} {improvement:+.1f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9.2 — Target Module Ablation: which projections matter most?
# ─────────────────────────────────────────────────────────────────────────────

target_configs = {
    'Q only':     ('q',),
    'V only':     ('v',),
    'Q+V':        ('q', 'v'),
    'Q+K+V':      ('q', 'k', 'v'),
    'Q+K+V+O':    ('q', 'k', 'v', 'o'),
}

target_results = {}
RANK = 8

for config_name, targets in target_configs.items():
    print(f"\n--- Training LoRA targets={config_name} ---")
    m = build_model(use_lora=True, rank=RANK, lora_targets=targets)
    hist = train(m, loader, n_epochs=ABLATION_EPOCHS, lr=3e-4, label=config_name)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    target_results[config_name] = {
        'history': hist,
        'final_loss': hist[-1],
        'params': n_params,
        'targets': targets,
    }

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('LoRA Target Module Ablation (rank=8)', fontsize=13, fontweight='bold')

colors2 = plt.cm.Set1(np.linspace(0, 0.8, len(target_configs)))

for (name, res), c in zip(target_results.items(), colors2):
    axes[0].plot(range(1, len(res['history'])+1), res['history'], '.-',
                color=c, label=name, linewidth=2, markersize=5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss by Target Projections')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

names = list(target_results.keys())
losses = [target_results[n]['final_loss'] for n in names]
params = [target_results[n]['params'] for n in names]

bars = axes[1].barh(names, losses, color=colors2, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Final Loss')
axes[1].set_title('Final Loss by Target Config')
axes[1].grid(True, alpha=0.3, axis='x')
for bar, (l, p) in zip(bars, zip(losses, params)):
    axes[1].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{l:.4f} ({p:,} params)', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('lora_target_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 10: Key Concepts Summary & Advanced Tips

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10.1 — Summary: Implementation Checklist
# ─────────────────────────────────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════╗
║             LoRA Implementation Checklist                    ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ✅ LoRALayer(in, out, rank, alpha)                          ║
║     - A: (rank × in)  — Kaiming uniform init                ║
║     - B: (out × rank) — Zeros init  (ΔW=0 at start)         ║
║     - scale = alpha / rank                                   ║
║     - forward: scale * x @ A.T @ B.T                        ║
║                                                              ║
║  ✅ LoRALinear (wraps nn.Linear)                             ║
║     - Freeze base weights (requires_grad=False)              ║
║     - forward: linear(x) + lora(x)                          ║
║     - merge_weights(): W_merged = W0 + scale*B@A             ║
║                                                              ║
║  ✅ Inject into Attention Q, K, V, O projections             ║
║     - Usually Q+V is sufficient (from original paper)        ║
║     - Skip K for efficiency if needed                        ║
║                                                              ║
║  ✅ Training setup                                           ║
║     - Only pass LoRA params to optimizer                     ║
║     - Can use higher LR than full fine-tune (3e-4 vs 1e-5)   ║
║     - AdamW + cosine schedule works well                     ║
║                                                              ║
║  ✅ Merging for inference                                    ║
║     - W_merged = W0 + (alpha/rank) * B @ A                   ║
║     - Zero inference overhead after merge                    ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  Hyperparameter Guidelines:                                  ║
║                                                              ║
║  rank (r):  4-8 for small tasks, 16-64 for complex tasks     ║
║  alpha:     Usually equal to rank (scale=1.0)               ║
║             Or 2*rank for stronger signal                    ║
║  dropout:   0.05-0.1 on A output (regularization)           ║
║  targets:   q+v (default), q+k+v+o (higher capacity)        ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  Extensions:                                                 ║
║                                                              ║
║  QLoRA:   Quantize base to 4-bit, LoRA adapters in float     ║
║  DoRA:    Decompose W into magnitude + direction             ║
║  LoRA+:   Different LRs for A and B matrices                 ║
║  AdaLoRA: Adaptive rank allocation per layer                 ║
║  LoRAHub: Compose multiple LoRA adapters                     ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10.2 — Bonus: QLoRA-style quantization wrapper sketch
# ─────────────────────────────────────────────────────────────────────────────

class QuantizedLoRALinear(nn.Module):
    """
    Sketch of QLoRA: the base weight is stored quantized (int8 here; 
    real QLoRA uses NF4), and LoRA adapters stay in float32.
    
    This shows the concept — full QLoRA needs bitsandbytes library.
    """
    def __init__(self, linear: nn.Linear, rank=8, alpha=16.0, bits=8):
        super().__init__()
        self.bits = bits
        
        # Quantize base weights to int8
        W = linear.weight.data.float()
        scale = W.abs().max() / (2**(bits-1) - 1)
        W_quant = (W / scale).round().clamp(-128, 127).to(torch.int8)
        
        # Store quantized weight (frozen, non-trainable)
        self.register_buffer('W_quant', W_quant)
        self.register_buffer('scale', torch.tensor(scale))
        self.bias = nn.Parameter(linear.bias.data.clone()) if linear.bias is not None else None
        
        # LoRA adapters stay in float32
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank=rank, alpha=alpha)
    
    def dequantize(self):
        """Dequantize weights for forward pass."""
        return self.W_quant.float() * self.scale
    
    def forward(self, x):
        # Dequantize on-the-fly
        W_float = self.dequantize()
        base_out = F.linear(x, W_float, self.bias)
        lora_out = self.lora(x)
        return base_out + lora_out


# Demo
print("QLoRA-style (int8 base + float LoRA):")
base_lin = nn.Linear(128, 128)
qlora = QuantizedLoRALinear(base_lin, rank=8, bits=8)

# Memory comparison
full_bytes  = base_lin.weight.numel() * 4   # float32
quant_bytes = qlora.W_quant.numel() * 1     # int8
lora_bytes  = sum(p.numel() * 4 for p in qlora.lora.parameters())

print(f"\n  Full float32 weight:    {full_bytes:,} bytes")
print(f"  Int8 quantized weight:  {quant_bytes:,} bytes  ({full_bytes/quant_bytes:.0f}x smaller)")
print(f"  LoRA adapters (float):  {lora_bytes:,} bytes")
print(f"  Total (quant+lora):     {quant_bytes+lora_bytes:,} bytes  ({full_bytes/(quant_bytes+lora_bytes):.1f}x vs full)")

x = torch.randn(2, 10, 128)
out = qlora(x)
print(f"\n  Output shape: {out.shape} ✅")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10.3 — Final Summary Visualization
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')
fig.patch.set_facecolor('#1a1a2e')

summary_text = [
    (0.5, 0.95, 'LoRA From Scratch — Summary', 20, 'white', 'bold'),
    (0.5, 0.88, 'Low-Rank Adaptation of Large Language Models (Hu et al., 2021)', 12, '#a0a0c0', 'normal'),
    
    (0.15, 0.78, 'CORE IDEA', 13, '#64b5f6', 'bold'),
    (0.15, 0.72, 'ΔW = B × A  where B∈R^(d×r), A∈R^(r×k)', 12, '#ffffff', 'normal'),
    (0.15, 0.67, 'h = W₀x + (α/r) × BAx', 12, '#ffffff', 'normal'),
    (0.15, 0.62, 'W₀ frozen, only A,B trained', 11, '#90caf9', 'normal'),
    
    (0.5, 0.78, 'KEY PROPERTIES', 13, '#81c784', 'bold'),
    (0.5, 0.72, '• B initialized to 0 → ΔW=0 at init', 11, '#ffffff', 'normal'),
    (0.5, 0.67, '• A initialized Kaiming uniform', 11, '#ffffff', 'normal'),
    (0.5, 0.62, '• Merge: W_merged = W₀ + (α/r)BA', 11, '#ffffff', 'normal'),
    (0.5, 0.57, '• Zero inference overhead after merge', 11, '#ffffff', 'normal'),
    
    (0.82, 0.78, 'PARAM SAVINGS', 13, '#ffb74d', 'bold'),
    (0.82, 0.72, 'Full: d×k params', 11, '#ffffff', 'normal'),
    (0.82, 0.67, 'LoRA: r(d+k) params', 11, '#ffffff', 'normal'),
    (0.82, 0.62, 'Savings: dk / r(d+k)', 11, '#ffffff', 'normal'),
    (0.82, 0.57, 'e.g. 768→768, r=8: 246×', 11, '#ffcc80', 'bold'),
    
    (0.25, 0.45, 'WHERE TO INJECT', 12, '#ce93d8', 'bold'),
    (0.25, 0.39, '✓ Q projection  (most impactful)', 10, '#ffffff', 'normal'),
    (0.25, 0.34, '✓ V projection  (standard choice)', 10, '#ffffff', 'normal'),
    (0.25, 0.29, '○ K projection  (optional)', 10, '#a0a0a0', 'normal'),
    (0.25, 0.24, '○ O projection  (optional)', 10, '#a0a0a0', 'normal'),
    
    (0.65, 0.45, 'HYPERPARAMETERS', 12, '#f48fb1', 'bold'),
    (0.65, 0.39, 'rank:    4-8 (small), 16-64 (large)', 10, '#ffffff', 'normal'),
    (0.65, 0.34, 'alpha:   = rank (scale=1) typical', 10, '#ffffff', 'normal'),
    (0.65, 0.29, 'dropout: 0.05-0.1 on lora path', 10, '#ffffff', 'normal'),
    (0.65, 0.24, 'lr:      3e-4 (10× higher than FFT)', 10, '#ffffff', 'normal'),
    
    (0.5, 0.13, 'Built entirely from scratch: LoRALayer → LoRALinear → LoRA Attention → MiniGPT', 11, '#80cbc4', 'italic'),
    (0.5, 0.07, 'Paper: "LoRA: Low-Rank Adaptation of Large Language Models" — Hu et al. (2021)', 10, '#607d8b', 'normal'),
]

for (x_pos, y_pos, text, size, color, weight) in summary_text:
    ax.text(x_pos, y_pos, text, transform=ax.transAxes,
            fontsize=size, color=color, fontweight=weight,
            ha='center', va='center',
            fontfamily='monospace' if '×' in text or 'ΔW' in text else 'sans-serif')

# Draw separator lines
for y in [0.82, 0.50, 0.17]:
    ax.axhline(y=y, color='#333355', linewidth=1, transform=ax.transAxes)

plt.tight_layout()
plt.savefig('lora_summary.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\n🎉 LoRA from scratch — complete!")
print("Files saved: lora_intuition.png, lora_training_results.png,")
print("             lora_analysis.png, lora_ablation.png,")
print("             lora_target_ablation.png, lora_summary.png")